# Scratch: Test VCP en Monedas (data diaria)

Notebook liviano para probar deteccion VCP en pares de monedas.
Solo muestra tablas de resultados, sin MLflow.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from models.configs import ATRZigZagConfig
from vcp_detection.heuristic import ATRZigZagDetector, run_full_vcp_pipeline
from vcp_detection.analysis import group_signals_into_patterns, simulate_trade

pd.set_option("display.float_format", "{:.4f}".format)
print("Imports OK")

## Configuracion

Modifica los parametros aca y re-ejecuta las celdas de abajo.
Los precios FX tienen rangos chicos (ej. EURUSD ~1.05-1.20), por lo que
los umbrales ATR y de profundidad pueden necesitar ajuste.

In [ ]:
USE_VOLUME_CONTRACTION = True
VOLUME_RATIO_THRESHOLD = 1.5

SWING_CONFIG = ATRZigZagConfig(atr_length=14, atr_mult=2.0, use_close_only=False)

SEQUENCE_PARAMS = {
    "method": "tolerance",
    "min_contractions": 2,
    "max_contractions": 6,
    "lookback_bars": 126,
    "tolerance": 0.10,
    "max_depth_pct": 0.35,
    "max_depth_atr": 5.0,
    "min_total_reduction": 0.80,
    "max_gap_between_contractions_days": None,
    "require_ascending_lows": True,
    "ascending_lows_tolerance": 0.0,
}

COMPRESSION_PARAMS = {
    "method": "ratio",
    "atr_period": 14,
    "ratio_threshold": 0.85,
}

VOLUME_CONTRACTION_PARAMS = {
    "method": "ratio",
    "volume_column": "volume",
    "ratio_threshold": 0.85,
}

BREAKOUT_PARAMS = {
    "volume_method": "ratio",
    "volume_ratio_threshold": VOLUME_RATIO_THRESHOLD,
    "volume_lookback_days": 50,
    "require_volume_confirmation": True,
    "max_entry_distance_pct": 0.05,
}

RISK_PARAMS = {
    "max_stop_loss_pct": 0.07,
    "breakeven_r_multiple": 2.0,
    "trailing_sma_period": 20,
    "trailing_volume_factor": 1.5,
    "trailing_stop_method": "atr",
    "trailing_atr_period": 14,
    "trailing_atr_multiplier": 3.0,
    "max_bars_without_progress": 20,
    "min_progress_r": 0.5,
    "early_exit_days": 3,
}

DATA_DIR = project_root / "data" / "monedas"
TICKERS = sorted([p.stem for p in DATA_DIR.glob("*.csv")])
print(f"Pares ({len(TICKERS)}): {TICKERS}")

## Ejecucion

In [ ]:
def load_ohlc(ticker):
    return pd.read_csv(DATA_DIR / f"{ticker}.csv", parse_dates=["date"], index_col="date")


vol_params = VOLUME_CONTRACTION_PARAMS if USE_VOLUME_CONTRACTION else None
rows = []

for ticker in TICKERS:
    ohlc = load_ohlc(ticker)
    detector = ATRZigZagDetector(SWING_CONFIG)

    results = run_full_vcp_pipeline(
        ohlc=ohlc,
        swing_detector=detector,
        sequence_params=SEQUENCE_PARAMS,
        compression_params=COMPRESSION_PARAMS,
        breakout_params=BREAKOUT_PARAMS,
        volume_contraction_params=vol_params,
    )

    signals = {dt: sig for dt, sig in results.items() if sig is not None}
    patterns = group_signals_into_patterns(signals, risk_params=RISK_PARAMS)

    trades = []
    for p in patterns:
        trade = simulate_trade(ohlc, p, RISK_PARAMS)
        trade["pattern"] = p
        trades.append(trade)

    n_trades = len(trades)
    wins = sum(1 for t in trades if t["pnl_pct"] > 0)
    cum_ret = float(np.prod([1 + t["pnl_pct"] for t in trades]) - 1) if trades else 0.0
    avg_r = float(np.mean([t["r_multiple"] for t in trades])) if trades else 0.0

    rows.append({
        "ticker": ticker,
        "n_signals": len(signals),
        "n_patterns": len(patterns),
        "n_trades": n_trades,
        "wins": wins,
        "losses": n_trades - wins,
        "win_rate": wins / n_trades if n_trades > 0 else 0.0,
        "cum_return": cum_ret,
        "avg_r": avg_r,
    })
    status = f"{n_trades}T ({wins}W/{n_trades - wins}L) CR={cum_ret:+.1%}" if n_trades > 0 else "--"
    print(f"  {ticker}: {status}")

print("\nListo.")

## Resultados

In [ ]:
df = pd.DataFrame(rows)

traded = df[df["n_trades"] > 0]
totals = pd.DataFrame([{
    "ticker": "TOTAL",
    "n_signals": int(df["n_signals"].sum()),
    "n_patterns": int(df["n_patterns"].sum()),
    "n_trades": int(df["n_trades"].sum()),
    "wins": int(df["wins"].sum()),
    "losses": int(df["losses"].sum()),
    "win_rate": float(traded["win_rate"].mean()) if len(traded) > 0 else 0.0,
    "cum_return": float(traded["cum_return"].mean()) if len(traded) > 0 else 0.0,
    "avg_r": float(traded["avg_r"].mean()) if len(traded) > 0 else 0.0,
}])
result = pd.concat([df, totals], ignore_index=True)

def color_returns(val):
    if isinstance(val, (int, float)):
        if val > 0: return "background-color: #27ae60; color: white"
        elif val < 0: return "background-color: #e74c3c; color: white"
    return ""

display(result.style.format({
    "win_rate": "{:.0%}",
    "cum_return": "{:+.1%}",
    "avg_r": "{:.2f}",
    "n_signals": "{:.0f}",
    "n_patterns": "{:.0f}",
    "n_trades": "{:.0f}",
    "wins": "{:.0f}",
    "losses": "{:.0f}",
}).map(color_returns, subset=["cum_return"]).background_gradient(
    subset=["win_rate"], cmap="RdYlGn", vmin=0, vmax=1
))